Markdown for commands

We are assuming aws key and secret in bashrc

export POLARIS_TOKEN=$(curl -s \
  "$POLARIS_URL/api/catalog/v1/oauth/tokens" \
  --user root:s3cr3t \
  -d 'grant_type=client_credentials' \
  -d 'scope=PRINCIPAL_ROLE:ALL' \
  | jq -r '.access_token')


echo $POLARIS_TOKEN


# create bucket olist

aws --endpoint-url http://localhost:9001 \
  s3 mb s3://olist


# create catalog called olist that point to olist s3 bucket

curl -s \
  -X POST "$POLARIS_URL/api/management/v1/catalogs" \
  -H "Authorization: Bearer $POLARIS_TOKEN" \
  -H "Content-Type: application/json" \
  -d '{
    "catalog": {
      "type": "INTERNAL",
      "name": "olist",
      "properties": {
        "default-base-location": "s3://olist/"
      },
      "storageConfigInfo": {
        "storageType": "S3",
        "allowedLocations": [
          "s3://olist/"
        ],
        "endpoint": "http://localhost:9001",
        "endpointInternal": "http://seaweedfs:8333",
        "region": "us-east-1",
        "pathStyleAccess": true,
        "stsUnavailable": true
      }
    }
  }' | jq


In [1]:
from pyspark.sql import SparkSession, functions as F
from decimal import Decimal

import os

polaris_url = os.environ.get("POLARIS_BASE_URL", "http://localhost:8181").rstrip("/")
catalog_name = os.environ.get("POLARIS_CATALOG", "olist")
polaris_credential = os.environ.get("POLARIS_CREDENTIAL", "root:s3cr3t") # polaris username/password
s3_endpoint = os.environ.get("S3_ENDPOINT", "http://localhost:9001")

spark = (
    SparkSession.builder
    .appName("Polaris-S3-HDFS-Hive")
    .master("spark://localhost:7077")

    # old legacy setup without iceberg, hive catalog
    .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.hive_catalog.type","hive")
    .config("spark.sql.catalog.hive_catalog.uri","thrift://localhost:9083")
    .config("spark.sql.catalog.hive_catalog.warehouse", "hdfs://localhost:9000/user/hive/warehouse")
        
    # ICEBERG
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")



        
    # HIVE & HADOOP START wtih iceberg
     
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")

    # Hive setup
    .config("spark.sql.catalog.iceberg.type", "hive")
    .config("spark.sql.catalog.iceberg.uri", "thrift://localhost:9083")
    # iceberg catalog points to HDFS
    .config("spark.sql.catalog.iceberg.warehouse", "hdfs://localhost:9000/user/hive/warehouse")

    # HDFS Warehouse, default warehouse, in case nothing is mentioned
    .config("spark.sql.warehouse.dir", "hdfs://localhost:9000/user/hive/warehouse")

    # HIVE & HADOOP END

    # POLARIS
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

     # SeaweedFS S3A: read existing bronze files
    .config("spark.hadoop.fs.s3a.endpoint", s3_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", "team")
    .config("spark.hadoop.fs.s3a.secret.key", "team1234")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

     # Polaris Iceberg REST catalog
    .config("spark.sql.catalog.polaris", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.polaris.type", "rest")
    .config("spark.sql.catalog.polaris.uri", f"{polaris_url}/api/catalog")
    # polaris catalog point to s3
    # while catalog in polaris using curl, you mention s3: location
    .config("spark.sql.catalog.polaris.warehouse", catalog_name) #olist
     # to get token
    .config("spark.sql.catalog.polaris.oauth2-server-uri", f"{polaris_url}/api/catalog/v1/oauth/tokens")
    .config("spark.sql.catalog.polaris.credential", polaris_credential)
    .config("spark.sql.catalog.polaris.scope", "PRINCIPAL_ROLE:ALL")

    # Iceberg storage: direct credentials, no STS credential vending
    .config("spark.sql.catalog.polaris.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.polaris.s3.endpoint", s3_endpoint)
    .config("spark.sql.catalog.polaris.s3.path-style-access", "true")
    .config("spark.sql.catalog.polaris.s3.access-key-id", "team")
    .config("spark.sql.catalog.polaris.s3.secret-access-key", "team1234")
    .config("spark.sql.catalog.polaris.client.region", "us-east-1")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
        
    .config("spark.sql.session.timeZone", "UTC")
    
    .config("spark.sql.shuffle.partitions", "4")

    # optional, disable metric exporter
    .config("spark.sql.catalog.polaris.rest-metrics-reporting-enabled", "false")
    .enableHiveSupport()
    .getOrCreate()
)

#spark.conf.set("spark.wap.id", "batch_20260929")

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

26/10/09 06:49:22 WARN Utils: Your hostname, dev-virtual-machine resolves to a loopback address: 127.0.1.1; using 10.199.100.80 instead (on interface ens160)
26/10/09 06:49:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 06:49:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.9
Spark UI: http://10.199.100.80:4040


In [2]:
spark.sql("SHOW CATALOGS").show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+



In [3]:
# we have iceberg as catalog configured in hive catalog
spark.sql("SHOW NAMESPACES in iceberg").show()

26/10/09 06:05:34 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist


+---------+
|namespace|
+---------+
|  default|
|     demo|
|  moviedb|
| training|
+---------+



In [4]:
spark.sql("SHOW CATALOGS").show()

+-------------+
|      catalog|
+-------------+
|      iceberg|
|spark_catalog|
+-------------+



In [5]:
# we have polaris (olist) as catalog configured in polaris catalog
spark.sql("SHOW NAMESPACES in polaris").show()

26/10/09 06:06:54 WARN AuthManagers: Inferring rest.auth.type=oauth2 since property credential was provided. Please explicitly set rest.auth.type to avoid this warning.


+---------+
|namespace|
+---------+
+---------+



In [7]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.olist_silver").show()

++
||
++
++



In [8]:
# we have polaris (olist) as catalog configured in polaris catalog
spark.sql("SHOW NAMESPACES in polaris").show()

+------------+
|   namespace|
+------------+
|olist_silver|
+------------+



In [9]:
spark.sql("SHOW CATALOGS").show()

+-------------+
|      catalog|
+-------------+
|      iceberg|
|      polaris|
|spark_catalog|
+-------------+



In [10]:
# create database in hdfs/ hive
# hdfs dfs -ls  /user/hive/warehouse

# this should list 

spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.customerdb").show()

26/10/09 06:13:01 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist


++
||
++
++



In [11]:
spark.sql("""
show databases in iceberg
""").show()


+----------+
| namespace|
+----------+
|customerdb|
|   default|
|      demo|
|   moviedb|
|  training|
+----------+



In [12]:
# iceberg.demo.customers, means it will use hdfs/hive

spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.customerdb.hdfs_hive_customers (
    customer_id BIGINT NOT NULL,
    name STRING,
    city STRING,
    amount DECIMAL(10,2)
) USING iceberg
TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet',
    'write.parquet.compression-codec'='zstd',
    'write.update.mode'='copy-on-write',
    'write.delete.mode'='copy-on-write',
    'write.merge.mode'='copy-on-write'
)
""")

DataFrame[]

In [13]:
spark.sql("""
show tables in iceberg.customerdb
""").show()

+----------+-------------------+-----------+
| namespace|          tableName|isTemporary|
+----------+-------------------+-----------+
|customerdb|hdfs_hive_customers|      false|
+----------+-------------------+-----------+



In [14]:
spark.sql("""
show databases in polaris
""").show()

+------------+
|   namespace|
+------------+
|olist_silver|
+------------+



In [15]:
# create table in polaris and create fiels in s3:/olist/..
spark.sql("""
CREATE TABLE IF NOT EXISTS polaris.olist_silver.s3_polaris_customers (
    customer_id BIGINT NOT NULL,
    name STRING,
    city STRING,
    amount DECIMAL(10,2)
) USING iceberg
TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet',
    'write.parquet.compression-codec'='zstd',
    'write.update.mode'='copy-on-write',
    'write.delete.mode'='copy-on-write',
    'write.merge.mode'='copy-on-write'
)
""")

DataFrame[]

In [16]:
# catalog.(namespace/database).tablename
spark.sql("""
show tables in polaris.olist_silver
""").show()

+------------+--------------------+-----------+
|   namespace|           tableName|isTemporary|
+------------+--------------------+-----------+
|olist_silver|s3_polaris_customers|      false|
+------------+--------------------+-----------+



In [17]:
# insert into polaris/s3
spark.sql("""
INSERT INTO polaris.olist_silver.s3_polaris_customers VALUES
    (1, 'Alice', 'Bangalore', CAST(1200 AS DECIMAL(10,2))),
    (2, 'Bob', 'Hyderabad', CAST(1800 AS DECIMAL(10,2))),
    (3, 'Carol', 'Chennai', CAST(950 AS DECIMAL(10,2)))
""")

26/10/09 06:20:34 WARN RESTMetricsReporter: Failed to report metrics to REST endpoint v1/olist/namespaces/olist_silver/tables/s3_polaris_customers/metrics
org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'root' with activated PrincipalRoles '[service_admin]' and activated grants via '[service_admin, catalog_admin]' is not authorized for op REPORT_WRITE_METRICS
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:214)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:240)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:336)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:297)
	at org.apache.iceberg.rest.BaseHTTPClient.post(BaseHTTPClient.java:100)
	at org.apache.iceberg.rest.RESTClient.post(RESTClient.java:126)
	at org.apache.iceberg.rest.RESTMetricsReporter.lambda$report$1(RESTMetricsReporte

DataFrame[]

In [18]:
# insert into hive/hdfs
spark.sql("""
INSERT INTO iceberg.customerdb.hdfs_hive_customers VALUES
    (1, 'Alice', 'Bangalore', CAST(1200 AS DECIMAL(10,2))),
    (2, 'Bob', 'Hyderabad', CAST(1800 AS DECIMAL(10,2))),
    (3, 'Carol', 'Chennai', CAST(950 AS DECIMAL(10,2)))
""")

26/10/09 06:21:24 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist


DataFrame[]

In [21]:
# from hive/hdfs
spark.sql("""
SELECT * FROM iceberg.customerdb.hdfs_hive_customers
""").show()

[Stage 2:>                                                          (0 + 1) / 1]

+-----------+-----+---------+-------+
|customer_id| name|     city| amount|
+-----------+-----+---------+-------+
|          1|Alice|Bangalore|1200.00|
|          2|  Bob|Hyderabad|1800.00|
|          3|Carol|  Chennai| 950.00|
+-----------+-----+---------+-------+



In [22]:
# from polaris/s3
spark.sql("""
SELECT * FROM polaris.olist_silver.s3_polaris_customers
""").show()

26/10/09 06:22:53 WARN RESTMetricsReporter: Failed to report metrics to REST endpoint v1/olist/namespaces/olist_silver/tables/s3_polaris_customers/metrics
org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'root' with activated PrincipalRoles '[service_admin]' and activated grants via '[service_admin, catalog_admin]' is not authorized for op REPORT_READ_METRICS
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:214)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:240)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:336)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:297)
	at org.apache.iceberg.rest.BaseHTTPClient.post(BaseHTTPClient.java:100)
	at org.apache.iceberg.rest.RESTClient.post(RESTClient.java:126)
	at org.apache.iceberg.rest.RESTMetricsReporter.lambda$report$1(RESTMetricsReporter

+-----------+-----+---------+-------+
|customer_id| name|     city| amount|
+-----------+-----+---------+-------+
|          1|Alice|Bangalore|1200.00|
|          2|  Bob|Hyderabad|1800.00|
|          3|Carol|  Chennai| 950.00|
+-----------+-----+---------+-------+



In [23]:
# 
#hive catalog
spark.sql("""
show databases in hive_catalog
""").show()

+----------+
| namespace|
+----------+
|customerdb|
|   default|
|      demo|
|   moviedb|
|  training|
+----------+



26/10/09 06:41:10 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist


In [26]:
# spark catalog csv/json/orc/others
 
# here you can create table/external table with csv/parquet/orc/json etc
spark.sql("""
show databases in spark_catalog
""").show()

+----------+
| namespace|
+----------+
|contactsdb|
|customerdb|
|   default|
|      demo|
|   moviedb|
|  training|
+----------+



26/10/09 06:47:32 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
